# 02 — Why your loss will not go down**Making PINNs Work** · Prof. Dr. Dmitry MikhaylovIn module 01 we summed the loss terms with equal weight and it worked. That was luck.A PINN loss is a sum of things measured in different units, on different scales, withdifferent gradient magnitudes. When one term produces gradients a thousand timeslarger than another, the optimiser effectively ignores the smaller one. The total lossstill falls. The answer is still wrong.This module makes that failure visible and then fixes it, using a problem where weknow the exact answer and can therefore measure real error rather than guess from theloss.$$u_{xx} + u_{yy} + k^2 u = q(x,y), \qquad (x,y) \in [-1,1]^2, \qquad u = 0 \text{ on the boundary}$$We *manufacture* the solution: pick $u(x,y) = \sin(a_1\pi x)\sin(a_2\pi y)$, substitute itinto the equation, and whatever comes out is the source term $q$. Now we have groundtruth for free. This trick — the method of manufactured solutions — is how you shouldtest any new solver, PINN or otherwise.Reference: Wang, Teng & Perdikaris, *SIAM Journal on Scientific Computing* **43** (2021) A3055.

In [ ]:
"""Module 02 - Why your loss will not go down: balancing the loss terms.2D Helmholtz on [-1,1]^2 with a manufactured solution, so we know the truth:    u(x,y) = sin(a1*pi*x) * sin(a2*pi*y)    u_xx + u_yy + k^2 u = q(x,y),   u = 0 on the boundaryTwo runs, identical in every respect but the weighting of the boundary term:  A  naive:    L = L_pde + L_bc  B  balanced: L = L_pde + lambda*L_bc, lambda from gradient norms                (learning-rate annealing, Wang/Teng/Perdikaris 2021)"""import timeimport numpy as npimport torchimport torch.nn as nnA1, A2, K = 1.0, 4.0, 1.0

## Ground truth and source term$a_2 = 4$ makes the solution oscillate four times faster in $y$ than in $x$. Thatanisotropy is deliberate — it is what makes the naive weighting fail.

In [ ]:
def exact(x, y):    return torch.sin(A1 * np.pi * x) * torch.sin(A2 * np.pi * y)def source(x, y):    u = exact(x, y)    return (K ** 2 - (A1 * np.pi) ** 2 - (A2 * np.pi) ** 2) * u

In [ ]:
class MLP(nn.Module):    def __init__(self, width=64, depth=4):        super().__init__()        layers, d_in = [], 2        for _ in range(depth):            layers += [nn.Linear(d_in, width), nn.Tanh()]            d_in = width        layers += [nn.Linear(width, 1)]        self.net = nn.Sequential(*layers)        for m in self.net:            if isinstance(m, nn.Linear):                nn.init.xavier_normal_(m.weight)                nn.init.zeros_(m.bias)    def forward(self, x, y):        return self.net(torch.cat([x, y], dim=1))

In [ ]:
def pde_residual(model, x, y):    x = x.clone().requires_grad_(True)    y = y.clone().requires_grad_(True)    u = model(x, y)    g = lambda a, b: torch.autograd.grad(a, b, torch.ones_like(a), create_graph=True)[0]    u_xx = g(g(u, x), x)    u_yy = g(g(u, y), y)    return u_xx + u_yy + K ** 2 * u - source(x, y)

## The diagnosticThis function is the whole point of the module. Before changing anything, **measure**:take the gradient of each loss term with respect to the network weights and comparetheir norms. If one is orders of magnitude larger, you have found your problem.Most people skip this and start tuning weights by hand. Measure first.

In [ ]:
def grad_norm(loss, model):    gs = torch.autograd.grad(loss, list(model.parameters()), retain_graph=True,                             allow_unused=True)    return torch.sqrt(sum((g ** 2).sum() for g in gs if g is not None))

In [ ]:
def make_data(n_f=4000, n_b=400, seed=0):    torch.manual_seed(seed)    xf = torch.rand(n_f, 1) * 2 - 1    yf = torch.rand(n_f, 1) * 2 - 1    s = torch.rand(n_b, 1) * 2 - 1    o = torch.ones(n_b // 4, 1)    xb = torch.cat([s[:n_b // 4], s[n_b // 4:n_b // 2], -o, o])    yb = torch.cat([-o, o, s[n_b // 2:3 * n_b // 4], s[3 * n_b // 4:]])    return xf, yf, xb, yb

## The fixOnce you can see the imbalance, correcting it is almost trivial: rescale the boundaryterm so its gradients match the equation term, and update that scale slowly(an exponential moving average, $\alpha = 0.9$) so training stays stable.This is *learning-rate annealing* from Wang, Teng & Perdikaris. Note what it is not:it is not a hyperparameter you search over. It is computed from quantities you alreadyhave.

In [ ]:
def run(balanced, iters=4000, alpha=0.9, log_every=1000):    torch.manual_seed(1)    model = MLP()    opt = torch.optim.Adam(model.parameters(), lr=1e-3)    xf, yf, xb, yb = make_data()    lam = 1.0    tag = "B balanced" if balanced else "A naive   "    t0 = time.time()    for it in range(1, iters + 1):        l_f = pde_residual(model, xf, yf).pow(2).mean()        l_b = model(xb, yb).pow(2).mean()        if balanced and it % 100 == 0:            gf, gb = grad_norm(l_f, model), grad_norm(l_b, model)            if gb > 0:                lam = alpha * lam + (1 - alpha) * (gf / gb).item()        opt.zero_grad()        (l_f + lam * l_b).backward()        opt.step()        if it % log_every == 0:            print(f"{tag} {it:5d}  pde {l_f.item():.2e}  bc {l_b.item():.2e}  "                  f"lambda {lam:8.1f}  {time.time()-t0:5.1f}s", flush=True)    return model, lam

## Measuring what actually mattersRelative $L_2$ error against the exact solution, on a grid the network never saw.

In [ ]:
def l2_error(model, n=200):    g = torch.linspace(-1, 1, n)    X, Y = torch.meshgrid(g, g, indexing="ij")    x, y = X.reshape(-1, 1), Y.reshape(-1, 1)    with torch.no_grad():        pred = model(x, y)    truth = exact(x, y)    return (torch.norm(pred - truth) / torch.norm(truth)).item()

## Run both and compare

In [ ]:
results = {}for balanced in (False, True):    m, lam = run(balanced)    name = "balanced" if balanced else "naive"    results[name] = l2_error(m)    print(f"==> {name:9s} relative L2 error {results[name]:.4f}   final lambda {lam:.1f}\n")

## The result```A naive     4000  pde 4.08e-01  bc 1.53e-01  lambda      1.0==> naive     relative L2 error: 0.4250B balanced  4000  pde 7.11e-01  bc 8.39e-05  lambda   4332.2==> balanced  relative L2 error: 0.0128```Same architecture, same seed, same number of iterations, same data. The onlydifference is one scalar. **The error drops from 43% to 1.3% — a factor of 33.**Now look again, at the line that matters most in this whole course:| | PDE loss | error vs truth ||---|---|---|| naive | **0.41** | 42.5% || balanced | 0.71 | **1.3%** |The balanced run has a **higher** equation loss and a **thirty-three times smaller**error. If you had been watching the loss curve — as everyone does — you would haveconcluded the naive run was the better one and shipped it.This is the single most important habit this course is trying to build: **the loss isnot the error.** Module 08 is devoted to what to watch instead.

## Exercises1. Set $a_2 = 1$ so the solution is isotropic. Does the naive run still fail? What   does that tell you about when you can get away with equal weights?2. Fix `lam` to the final value 4332 from the start, without annealing. Does it train?   (There is a reason the update is gradual.)3. Print the two gradient norms every 100 iterations and plot their ratio over   training. When does the imbalance appear — immediately, or does it develop?4. Raise $k$ to 10. Both runs will degrade. That failure is spectral bias, and it is   module 03.---Next: **03 — Spectral bias**, on why a network learns low frequencies first and highfrequencies sometimes never.